### This file demonstrates the usage of box-jenkins methodology


In [ ]:
# import the necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# load the dataset
df = pd.read_csv("../data/electricity_load.csv")
df.head()

In [ ]:
# prepare the time series
df["datetime"] = pd.to_datetime(
    df["datetime"],
    format="%d/%m/%Y %H:%M"
)
df = df.sort_values("datetime")
df = df.set_index("datetime")
df.head()

In [ ]:
series = df["demand_kWh"].copy()

print(series.describe())
print("Missing values:", series.isna().sum())

In [ ]:
series = series.interpolate(method="time")

series = series.ffill().bfill()

print("Missing values after interpolation:", series.isna().sum())

### EDA

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(series)
plt.title("Hourly Electricity Load")
plt.xlabel("Datetime")
plt.ylabel("Load (kWh)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# plot first few days
plt.figure(figsize=(14, 5))

plt.plot(series.iloc[:24 * 7])

plt.title("Electricity Load — First 7 Days")
plt.xlabel("Hour")
plt.ylabel("Load (kWh)")

plt.grid(True)
plt.tight_layout()
plt.show()

### split into train test dataset

In [ ]:
train_size = int(len(series) * 0.80)
train = series.iloc[:train_size]
test = series.iloc[train_size:]

print("Training observations:", len(train))
print("Testing observations :", len(test))

In [ ]:
# visualize the splits
plt.figure(figsize=(14, 5))

plt.plot(train.index, train, label="Training")
plt.plot(test.index, test, label="Testing")

plt.title("Training and Testing Data")
plt.xlabel("Datetime")
plt.ylabel("Load (kWh)")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### BOX–JENKINS STEP 1 — IDENTIFICATION

The purpose of identification is to determine:

$$ (p,d,q)(P,D,Q)_s $$

For our hourly electricity data:

$$ s = 24 $$

because one seasonal cycle corresponds approximately to 24 hours.

In [ ]:
# check stationarity using ADF test
def adf_test(series, name="Series"):
    result = adfuller(series.dropna())

    print(f"ADF Test: {name}")
    print("=" * 40)

    print(f"ADF Statistic : {result[0]:.6f}")
    print(f"p-value       : {result[1]:.6f}")

    print("\nCritical Values:")

    for key, value in result[4].items():
        print(f"  {key}: {value:.6f}")

    if result[1] < 0.05:
        print("\nResult: Stationary")
    else:
        print("\nResult: Non-stationary")

In [ ]:
# run the test: 
adf_test(train, "Original Electricity Load")

### | Statistic          |          Value |
| ------------------ | -------------: |
| ADF statistic      | **−12.358840** |
| p-value            |    **< 0.001** |
| 1% critical value  |      −3.430818 |
| 5% critical value  |      −2.861747 |
| 10% critical value |      −2.566880 |
H
0
	​
=The series has a unit root (non-stationary)
$$ H_1 = \text{The series is stationary} $$
ADF statistic is −12.36, which is much smaller than the 5% critical value of −2.86.
Also:
$$ p < 0.05 $$
Therefore, we reject \(H_0\).
Conclusion
The original electricity-load series is stationary according to the ADF test.
Therefore, for the non-seasonal component:
$$ \boxed{d=0} $$
So do not apply ordinary first-order differencing.
Original electricity load
          -->
       ADF test
          -->
     Stationary
          -->
        d = 0
       -->
     Examine ACF
     Examine PACF
       -->
 Identify p and q
          -->
 Examine seasonal pattern
         -->
 Identify P, D, Q

But there is one more question: seasonal differencing \(D\)

The ADF result tells us that:

$$ d=0 $$

It does not automatically tell us that \(D=0\).

For hourly electricity demand, we need to investigate the seasonal period:

$$ s=24 $$

So now let's test whether seasonal differencing is necessary.

In [ ]:
seasonal_diff = train.diff(24).dropna()

adf_test(
    seasonal_diff,
    "Seasonally Differenced Electricity Load"
)

| Test                                 | ADF statistic | p-value | Conclusion |
| ------------------------------------ | ------------: | ------: | ---------- |
| Original series                      |      −12.3588 | < 0.001 | Stationary |
| Seasonal difference \(y_t-y_{t-24}\) |      −22.1394 | < 0.001 | Stationary |
Since the original series is already stationary, we should start with \(D=0\) rather than seasonally differencing unnecessarily.

So our initial SARIMA structure should be:

SARIMA(p,0,q)(P,0,Q)24
###
The next—and very important—step: ACF and PACF

Now we need to identify the AR and MA components.

Since \(d=0\), use the original training series, not the differenced series:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

plot_acf(
    train,
    lags=72,
    ax=axes[0]
)

axes[0].set_title("ACF — Original Electricity Load")

plot_pacf(
    train,
    lags=72,
    ax=axes[1],
    method="ywm"
)

axes[1].set_title("PACF — Original Electricity Load")

plt.tight_layout()
plt.show()

What  ACF tells us
Your ACF has a very clear pattern:
Very high correlation at lag 1.
Gradual decay through the early lags.
Strong seasonal peak around lag 24.
Another peak around lag 48.
Another smaller peak around lag 72.
This is strong evidence of 24-hour seasonality:
$$ s=24 $$
The fact that the ACF gradually tails off rather than cutting off sharply is characteristic of an AR component rather than a pure MA component.

So the ACF suggests:
$$ p > 0 $$
and
P>0

What the PACF tells us
This is particularly interesting.
PACF has:
Very strong spike at lag 1
Significant negative spike at lag 2
Then most of the early lags are relatively small
A noticeable seasonal spike at lag 24
This suggests a non-seasonal AR component around:
$$ p=2 $$
because the PACF has substantial effects at lags 1 and 2 and then largely settles down.
Therefore:
$$ \boxed{p=2} $$
The seasonal spike around lag 24 suggests that a seasonal AR or MA component should be considered.

Our Box–Jenkins identification

Putting your ADF + ACF + PACF results together:
| Component | Evidence                           |  Initial choice |
| --------- | ---------------------------------- | --------------: |
| \(d\)     | Original series is stationary      |           **0** |
| \(s\)     | Strong daily pattern               |          **24** |
| \(p\)     | PACF significant at lags 1–2       |           **2** |
| \(q\)     | ACF does not show a clear cutoff   | **0 initially** |
| \(P\)     | Seasonal structure around lag 24   | **1 candidate** |
| \(Q\)     | Seasonal ACF peak                  | **1 candidate** |
| \(D\)     | Original series already stationary | **0 initially** |
So a very natural first model is:

$$ \boxed{SARIMA(2,0,0)(1,0,0)_{24}} $$

which is exactly one of the models you had previously proposed.

However, the seasonal ACF pattern means we should also test:

$$ \boxed{SARIMA(2,0,0)(0,0,1)_{24}} $$

So the two candidate models are actually quite defensible from the plots:

In [ ]:
sarima_candidates = [
    {
        "name": "SARIMA(2,0,0)(1,0,0)[24]",
        "order": (2, 0, 0),
        "seasonal_order": (1, 0, 0, 24)
    },
    {
        "name": "SARIMA(2,0,0)(0,0,1)[24]",
        "order": (2, 0, 0),
        "seasonal_order": (0, 0, 1, 24)
    }
]

Box–Jenkins estimation

At this point, I recommend that you stop changing the identification and move to estimation.

Run the following complete cell:

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pandas as pd

sarima_candidates = [
    {
        "name": "SARIMA(2,0,0)(1,0,0)[24]",
        "order": (2, 0, 0),
        "seasonal_order": (1, 0, 0, 24)
    },
    {
        "name": "SARIMA(2,0,0)(0,0,1)[24]",
        "order": (2, 0, 0),
        "seasonal_order": (0, 0, 1, 24)
    }
]

results = []

for candidate in sarima_candidates:

    print("=" * 70)
    print(candidate["name"])
    print("=" * 70)

    model = SARIMAX(
        train,
        order=candidate["order"],
        seasonal_order=candidate["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_model = model.fit(disp=False)

    results.append({
        "name": candidate["name"],
        "model": fitted_model,
        "aic": fitted_model.aic,
        "bic": fitted_model.bic
    })

    print(f"AIC: {fitted_model.aic:.4f}")
    print(f"BIC: {fitted_model.bic:.4f}")
    print()

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": result["name"],
        "AIC": result["aic"],
        "BIC": result["bic"]
    }
    for result in results
])

comparison.sort_values("AIC")

### BOX–JENKINS STEP 2 — ESTIMATION

The candidate SARIMA models have now been estimated using the training data.

We compare the models using AIC and BIC. Lower values indicate a better balance between model fit and model complexity.

However, information criteria alone are not sufficient. The selected model must also pass residual diagnostic tests.


In [ ]:
# Select the model with the lowest AIC
best_result = min(results, key=lambda x: x["aic"])
best_model = best_result["model"]

print("Selected model based on AIC:")
print(best_result["name"])
print(f"AIC: {best_result['aic']:.4f}")
print(f"BIC: {best_result['bic']:.4f}")

print("\nModel summary:")
print(best_model.summary())


### BOX–JENKINS STEP 3 — DIAGNOSTIC CHECKING

A good SARIMA model should leave residuals that behave approximately like white noise.

We therefore check:

1. Residual time series
2. Residual ACF
3. Ljung–Box test for remaining autocorrelation

The desired result is that the residual autocorrelations are small and the Ljung–Box test does not reject the null hypothesis of no autocorrelation.


In [ ]:
# Extract residuals
residuals = best_model.resid.dropna()

print("Residual summary:")
print(residuals.describe())


In [ ]:
# Plot residuals and their ACF
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

axes[0].plot(residuals)
axes[0].set_title("Residuals of Selected SARIMA Model")
axes[0].set_xlabel("Datetime")
axes[0].set_ylabel("Residual")
axes[0].grid(True)

plot_acf(
    residuals,
    lags=72,
    ax=axes[1]
)
axes[1].set_title("ACF of SARIMA Residuals")

plt.tight_layout()
plt.show()


In [ ]:
# Ljung-Box test
# H0: residuals are independently distributed (no serial correlation)
lb_test = acorr_ljungbox(
    residuals,
    lags=[12, 24, 48],
    return_df=True
)

print("Ljung–Box test:")
display(lb_test)

alpha = 0.05

if (lb_test["lb_pvalue"] > alpha).all():
    print(
        "\nConclusion: Fail to reject H0 at all tested lags. "
        "The residuals are consistent with white noise."
    )
else:
    print(
        "\nConclusion: At least one tested lag has p < 0.05. "
        "Some residual autocorrelation remains, so the model may need refinement."
    )


### Residual diagnostic conclusion

The diagnostic plots and Ljung–Box test are used together.

- If residual ACF spikes are mostly inside the confidence bounds and Ljung–Box p-values are above 0.05, the model is considered adequate.
- If significant residual autocorrelation remains, the model should be reconsidered by changing the non-seasonal or seasonal AR/MA terms.

This is the diagnostic-checking stage of the Box–Jenkins methodology.


### BOX–JENKINS STEP 4 — FORECASTING

After selecting and diagnosing the model, we forecast the complete test period.

The model is fitted only on the training data, so the test set remains unseen during model estimation. This allows us to evaluate genuine out-of-sample forecasting performance.


In [ ]:
# Forecast the entire test period
forecast_result = best_model.get_forecast(steps=len(test))

forecast = forecast_result.predicted_mean
confidence_intervals = forecast_result.conf_int()

# Make sure the forecast uses the same datetime index as the test set
forecast.index = test.index
confidence_intervals.index = test.index

print("Forecast observations:", len(forecast))
print("\nFirst forecasted values:")
display(forecast.head())


In [ ]:
# Plot actual test values against SARIMA forecasts
plt.figure(figsize=(14, 6))

plt.plot(train.index, train, label="Training")
plt.plot(test.index, test, label="Actual Test")
plt.plot(forecast.index, forecast, label="SARIMA Forecast")

plt.fill_between(
    confidence_intervals.index,
    confidence_intervals.iloc[:, 0],
    confidence_intervals.iloc[:, 1],
    alpha=0.2,
    label="95% Confidence Interval"
)

plt.title(f"Electricity Load Forecast — {best_result['name']}")
plt.xlabel("Datetime")
plt.ylabel("Load (kWh)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


### BOX–JENKINS STEP 5 — FORECAST EVALUATION

We evaluate the forecasts against the actual test observations using:

- MAE — Mean Absolute Error
- RMSE — Root Mean Squared Error

Lower values indicate better forecasting performance.


In [ ]:
# Calculate forecast accuracy
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))

evaluation = pd.DataFrame({
    "Metric": ["MAE", "RMSE"],
    "Value": [mae, rmse]
})

display(evaluation)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")


In [ ]:
# Compare actual and forecast values
forecast_comparison = pd.DataFrame({
    "Actual": test,
    "Forecast": forecast,
    "Error": test - forecast,
    "Absolute_Error": (test - forecast).abs()
})

display(forecast_comparison.head(20))


### Final Box–Jenkins conclusion

The complete workflow is:

**Identification → Estimation → Diagnostic Checking → Forecasting → Evaluation**

For this hourly electricity-load series:

- The seasonal period is **s = 24**, representing the daily cycle.
- The original series was found to be stationary by the ADF test, giving **d = 0**.
- The ACF and PACF suggested a non-seasonal AR component, with **p = 2** used in the initial candidates.
- Two seasonal SARIMA candidates were estimated:
  - SARIMA(2,0,0)(1,0,0)[24]
  - SARIMA(2,0,0)(0,0,1)[24]
- The model with the lower AIC was selected as the initial best model.
- Residual diagnostics were then performed using residual plots, residual ACF, and the Ljung–Box test.
- Finally, the selected model was used to forecast the held-out test data and evaluated using MAE and RMSE.

If the residual diagnostics indicate significant remaining autocorrelation, the model should not be treated as final; return to the identification stage and test alternative p, q, P, and Q combinations.
